In [12]:
import numpy as np
import cvxpy as cp
from numpy.linalg import inv
from pillow_lab_rotation.tools import vec
from pillow_lab_rotation.lds import LinearDynamicalSystem
from pillow_lab_rotation.eirnn import EIRNNInit

from scipy.linalg import block_diag

import matplotlib.pyplot as plt

In [ ]:
regions_and_neurons = {
    'region_1' : {
        'Di' : 2,
        'De' : 2,
        'Ni' : 10,
        'Ne' : 10
    },
    'region_2' : {
        'Di' : 3,
        'De' : 3,
        'Ni' : 10,
        'Ne' : 10
    }
}

Cs = []
As = []
for region in regions_and_neurons.values():
    print(region)
    Di = region['Di']
    De = region['De']
    Ni = region['Ni']
    Ne = region['Ne']

    e2e_block = np.random.uniform(0, 1, (Ne, De))
    e2i_block = np.zeros((Ni, De))
    i2i_block = np.random.uniform(0, 1, (Ni, Di))
    i2e_block = np.zeros((Ne, Di))
    Cs.append(np.block(
        [[e2e_block, i2e_block],
         [e2i_block, i2i_block]]
    ))

    As.append(np.random.standard_normal((Di + De, Di + De)))


C = block_diag(*Cs)
A = block_diag(*As)

{'Di': 2, 'De': 2, 'Ni': 10, 'Ne': 10}
{'Di': 3, 'De': 3, 'Ni': 10, 'Ne': 10}


In [26]:
total_latents = 0
total_neurons = 0
for region in regions_and_neurons.values():
    total_latents += region['De'] + region['Di']
    total_neurons += region['Ne'] + region['Ni']


In [ ]:
class MRCTDS(LinearDynamicalSystem):
    def __init__(
            self,
            region_parameters: dict[dict],
            udim: int = 0
    ):
        self.region_parameters = region_parameters

        # Get useful numbers
        self.D = 0
        self.N = 0
        for region in regions_and_neurons.values():
            self.D += region['De'] + region['Di']
            self.N += region['Ne'] + region['Ni']


        # Initialize the parent LDS with inputs to latents only (no feedthrough)
        super().__init__(xdim=self.D, ydim=self.N, udim=udim, feedthrough=False)
        self.init_constraints(self)


    def init_constraints(self):
        NotImplemented

    def init_params(
            self,
            observations: np.ndarray|None = None,
            start_seed: int=0
    ):
        
        self.mu0 = np.random.standard_normal((self.D, 1))
        self.Q0 = np.eye(self.D)
        self.Q = np.eye(self.D)
        self.B = np.random.randn(self.D, self.udim) if self.udim > 0 else np.zeros((self.D, self.udim))

        Cs = []
        As = []
        for region in self.region_parameters.values():
            Di = region['Di']
            De = region['De']
            Ni = region['Ni']
            Ne = region['Ne']

            e2e_block = np.random.uniform(0, 1, (Ne, De))
            e2i_block = np.zeros((Ni, De))
            i2i_block = np.random.uniform(0, 1, (Ni, Di))
            i2e_block = np.zeros((Ne, Di))
            Cs.append(np.block(
                [[e2e_block, i2e_block],
                [e2i_block, i2i_block]]
            ))

            As.append(np.random.standard_normal((Di + De, Di + De)))

        self.C = block_diag(*Cs)
        self.A = block_diag(*As)

